In [0]:
# =========================================================
# FILE    : 04_utils.py
# PURPOSE : Shared Functions — Silver aur Gold dono use karein
# =========================================================

from delta.tables import DeltaTable
import logging

logger = logging.getLogger("ecommerce_pipeline.utils")

# =========================================================
# UPSERT FUNCTION — Sirf Ek Baar Likha, Kahin bhi Use Karo
# =========================================================

def upsert_delta(spark, df, path, merge_condition):
    """
    Smart Write Function:
    - Table exist kare  → MERGE (update ya insert)
    - Table na ho       → Pehli baar seedha likho

    Parameters:
        spark          : Spark Session
        df             : Jo data likhna hai
        path           : Kahan likhna hai
        merge_condition: Kaise match karein (e.g. "target.order_id = source.order_id")
    """

    if DeltaTable.isDeltaTable(spark, path):

        logger.info(f"Table exist karti hai → MERGE kar rahe hain : {path}")

        delta_table = DeltaTable.forPath(spark, path)

        (
            delta_table.alias("target")
            .merge(df.alias("source"), merge_condition)
            .whenMatchedUpdateAll()       # Match hua → Update karo
            .whenNotMatchedInsertAll()    # Naya hai  → Insert karo
            .execute()
        )

    else:

        logger.info(f"Table nahi hai → Pehli baar likh rahe hain : {path}")

        df.write.format("delta").save(path)

    logger.info("Upsert Completed")